# Cluster Forecasting Experiment Runner

This notebook keeps the forecasting features built from raw 2023 daily usage data.
Cluster labels are only used to route households into cluster-specific models.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_NAME = "Clustering-And-Forecasting-TimeSeries-PlayingGround"
REPO_URL = "https://github.com/QuirkyCroissant/Clustering-And-Forecasting-TimeSeries-PlayingGround.git"
BRANCH = os.environ.get("NOTEBOOK_GIT_BRANCH", "feat/cluster-forecasting-lgbm-xgb")

kaggle_root = Path("/kaggle/working")
cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME and (cwd / "src" / "forecasting").exists():
    REPO_ROOT = cwd
elif kaggle_root.exists():
    repo_dir = kaggle_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    REPO_ROOT = repo_dir.resolve()
else:
    raise RuntimeError(
        "Could not determine REPO_ROOT automatically. Open the notebook from the repo root or run it on Kaggle."
    )

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH)

Cloning into '/kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround'...


REPO_ROOT: /kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround
SRC_PATH: /kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround/src


Updating files: 100% (64/64), done.


In [2]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

from forecasting.data import (
    discover_cluster_cases,
    ensure_experiment_dirs,
    ensure_output_dirs,
    load_wide_csv,
)

OUT = ensure_output_dirs(REPO_ROOT)

TRAIN_23_PATH = REPO_ROOT / "data" / "raw" / "sample_23.csv"
TEST_24_PATH = REPO_ROOT / "data" / "raw" / "sample_24.csv"
CLUSTER_CASE_DIR = REPO_ROOT / "notebooks" / "outputs" / "feature"
SHAPELET_STATIC_PATH = REPO_ROOT / "notebooks" / "outputs" / "shapelet_experiments" / "shapelet_features.csv"

In [ ]:
DEBUG = False
DEBUG_FRAC = 0.2

MODEL_NAME = "xgb"
GPU_ENABLED = True
MIN_CLUSTER_ROWS = 500
MIN_CLUSTER_HOUSEHOLDS = 30

SELECTED_CASES = ["case5"]
INCLUDE_BASE_VARIANT = True
INCLUDE_SHAPELET_STATIC_VARIANT = True

PLOT_SAMPLE_PER_GROUP = 2
PLOT_MAX_GROUPS = None
RANDOM_STATE = 42

In [4]:
def build_experiment_configs(case_paths: dict[str, Path]) -> list[dict]:
    case_names = SELECTED_CASES or list(case_paths)
    missing = sorted(set(case_names) - set(case_paths))
    if missing:
        raise ValueError(f"Unknown case names requested: {missing}")

    variant_defs = []
    if INCLUDE_BASE_VARIANT:
        variant_defs.append(("base", None))
    if INCLUDE_SHAPELET_STATIC_VARIANT:
        if not SHAPELET_STATIC_PATH.exists():
            raise FileNotFoundError(f"Missing shapelet static feature file: {SHAPELET_STATIC_PATH}")
        variant_defs.append(("shapelet_static", SHAPELET_STATIC_PATH))

    configs = []
    for case_name in case_names:
        for variant_name, static_path in variant_defs:
            configs.append(
                {
                    "case_name": case_name,
                    "variant_name": variant_name,
                    "experiment_name": f"{case_name}_{variant_name}",
                    "cluster_path": str(case_paths[case_name]),
                    "static_features_path": str(static_path) if static_path else None,
                }
            )
    return configs


def build_run_settings() -> dict:
    return {
        "debug": DEBUG,
        "debug_frac": DEBUG_FRAC,
        "model_name": MODEL_NAME,
        "gpu_enabled": GPU_ENABLED,
        "min_cluster_rows": MIN_CLUSTER_ROWS,
        "min_cluster_households": MIN_CLUSTER_HOUSEHOLDS,
        "plot_sample_per_group": PLOT_SAMPLE_PER_GROUP,
        "plot_max_groups": PLOT_MAX_GROUPS,
        "random_state": RANDOM_STATE,
    }


def run_experiment_subprocess(exp_config: dict, run_settings: dict):
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_PATH) if not existing_pythonpath else str(SRC_PATH) + os.pathsep + existing_pythonpath

    cmd = [
        sys.executable,
        "-m",
        "forecasting.experiment",
        "--repo-root",
        str(REPO_ROOT),
        "--config-json",
        json.dumps(exp_config),
        "--settings-json",
        json.dumps(run_settings),
    ]

    try:
        subprocess.run(cmd, check=True, cwd=REPO_ROOT, env=env)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Experiment failed: {exp_config['experiment_name']}") from exc


def load_experiment_metric(exp_name: str, table_name: str) -> pd.DataFrame:
    metric_path = ensure_experiment_dirs(REPO_ROOT, exp_name)["metrics"] / f"{table_name}.csv"
    if not metric_path.exists():
        raise FileNotFoundError(f"Missing metric output for {exp_name}: {metric_path}")
    return pd.read_csv(metric_path)

In [5]:
cluster_case_paths = discover_cluster_cases(CLUSTER_CASE_DIR)
experiment_configs = build_experiment_configs(cluster_case_paths)
run_settings = build_run_settings()

experiment_plan = pd.DataFrame(experiment_configs)
display(experiment_plan)
print(f"Planned experiments: {len(experiment_configs)}")

,case_name,variant_name,experiment_name,cluster_path,static_features_path
0,case1,base,case1_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None
1,case1,shapelet_static,case1_shapelet_static,/kaggle/working/Clustering-And-Forecasting-Tim...,/kaggle/working/Clustering-And-Forecasting-Tim...
2,case2,base,case2_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None
3,case2,shapelet_static,case2_shapelet_static,/kaggle/working/Clustering-And-Forecasting-Tim...,/kaggle/working/Clustering-And-Forecasting-Tim...
4,case3,base,case3_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None
5,case3,shapelet_static,case3_shapelet_static,/kaggle/working/Clustering-And-Forecasting-Tim...,/kaggle/working/Clustering-And-Forecasting-Tim...
6,case4,base,case4_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None
7,case4,shapelet_static,case4_shapelet_static,/kaggle/working/Clustering-And-Forecasting-Tim...,/kaggle/working/Clustering-And-Forecasting-Tim...
8,case5,base,case5_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None
9,case5,shapelet_static,case5_shapelet_static,/kaggle/working/Clustering-And-Forecasting-Tim...,/kaggle/working/Clustering-And-Forecasting-Tim...


Planned experiments: 12


In [6]:
completed_experiments = []

for exp_config in experiment_configs:
    print(f"\n=== Running {exp_config['experiment_name']} ===")
    run_experiment_subprocess(exp_config, run_settings)
    completed_experiments.append(exp_config["experiment_name"])


=== Running case1_base ===


Building training rows: 100%|██████████| 17547/17547 [00:11<00:00, 1514.47it/s]


[0]	validation_0-mae:7.44852
[50]	validation_0-mae:2.75369
[100]	validation_0-mae:2.51175
[150]	validation_0-mae:2.49126
[200]	validation_0-mae:2.48443
[250]	validation_0-mae:2.48490
[300]	validation_0-mae:2.48403
[350]	validation_0-mae:2.48302
[399]	validation_0-mae:2.48132


Forecasting global:   0%|          | 0/366 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:23:40] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Forecasting by cluster: 100%|██████████| 366/366 [00:32<00:00, 11.22it/s]



=== Running case1_shapelet_static ===


...

Forecasting global:   0%|          | 0/366 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:49:47] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Forecasting by cluster: 100%|██████████| 366/366 [00:33<00:00, 11.06it/s]



=== Running case6_shapelet_static ===


Building training rows: 100%|██████████| 17547/17547 [00:11<00:00, 1520.51it/s]


[0]	validation_0-mae:7.44877
[50]	validation_0-mae:2.74677
[100]	validation_0-mae:2.48625
[150]	validation_0-mae:2.46517
[200]	validation_0-mae:2.46016
[250]	validation_0-mae:2.45561
[300]	validation_0-mae:2.45039
[350]	validation_0-mae:2.44702
[399]	validation_0-mae:2.44491


Forecasting global:   0%|          | 0/366 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:52:21] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Forecasting by cluster: 100%|██████████| 366/366 [00:40<00:00,  8.98it/s]


In [7]:
all_overall_summary = pd.concat(
    [load_experiment_metric(exp_name, "overall_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_route_summary = pd.concat(
    [load_experiment_metric(exp_name, "route_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_mae_summary = pd.concat(
    [load_experiment_metric(exp_name, "cluster_mae_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_compare_summary = pd.concat(
    [load_experiment_metric(exp_name, "cluster_compare_summary") for exp_name in completed_experiments],
    ignore_index=True,
)

model_comparison = (
    all_overall_summary
    .pivot(index="experiment_name", columns="model", values="mean_mae")
    .reset_index()
)
model_comparison["delta_cluster_minus_global"] = (
    model_comparison[f"cluster_{MODEL_NAME}"] - model_comparison[f"global_{MODEL_NAME}"]
)
model_comparison = model_comparison.sort_values("delta_cluster_minus_global").reset_index(drop=True)

all_overall_summary.to_csv(OUT["metrics"] / "all_experiments_overall_summary.csv", index=False)
all_route_summary.to_csv(OUT["metrics"] / "all_experiments_route_summary.csv", index=False)
all_cluster_mae_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_mae_summary.csv", index=False)
all_cluster_compare_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_compare_summary.csv", index=False)
model_comparison.to_csv(OUT["metrics"] / "all_experiments_model_comparison.csv", index=False)

display(all_overall_summary.sort_values(["experiment_name", "model"]).reset_index(drop=True))
display(model_comparison)
display(all_cluster_compare_summary.head(20))

,n_households,mean_mae,median_mae,std_mae,model,experiment_name
0,17547,4.749027,2.580197,6.564125,cluster_xgb,case1_base
1,17547,4.827977,2.411192,7.332080,global_xgb,case1_base
2,17547,4.841055,2.972356,6.338637,cluster_xgb,case1_shapelet_static
3,17547,5.846882,2.640865,7.774905,global_xgb,case1_shapelet_static
4,17547,4.643795,2.825900,5.785807,cluster_xgb,case2_base
5,17547,4.827977,2.411192,7.332080,global_xgb,case2_base
6,17547,4.982278,2.902687,6.168969,cluster_xgb,case2_shapelet_static
7,17547,5.846882,2.640865,7.774905,global_xgb,case2_shapelet_static
8,17547,4.739561,2.635222,7.028448,cluster_xgb,case3_base
9,17547,4.827977,2.411192,7.332080,global_xgb,case3_base


model,experiment_name,cluster_xgb,global_xgb,delta_cluster_minus_global
0,case1_shapelet_static,4.841055,5.846882,-1.005827
1,case2_shapelet_static,4.982278,5.846882,-0.864605
2,case4_shapelet_static,5.215500,5.846882,-0.631382
3,case3_shapelet_static,5.231505,5.846882,-0.615378
4,case6_shapelet_static,5.245627,5.846882,-0.601256
5,case5_shapelet_static,5.261631,5.846882,-0.585251
6,case6_base,4.514706,4.827977,-0.313271
7,case5_base,4.618317,4.827977,-0.209660
8,case4_base,4.635950,4.827977,-0.192028
9,case2_base,4.643795,4.827977,-0.184182


,ForecastGroup,RefinedCluster,n_households,mean_mae_global,mean_mae_cluster,mean_delta,median_delta,experiment_name
0,4,4,84,15.102638,11.736138,-3.366499,-0.453219,case1_base
1,2,2,1916,15.083532,13.132751,-1.950781,-0.390800,case1_base
2,8,8,51,8.634279,7.971798,-0.662480,-0.498981,case1_base
3,1,1,2091,7.959311,7.358744,-0.600566,-0.011100,case1_base
4,3,3,280,2.798734,2.545652,-0.253082,0.024280,case1_base
5,7,7,198,1.603658,1.565934,-0.037724,0.005359,case1_base
6,0,0,12641,2.843474,3.013328,0.169854,0.042697,case1_base
7,6,6,46,5.226070,7.425635,2.199565,0.151182,case1_base
8,5,5,86,1.933299,22.337390,20.404091,0.311545,case1_base
9,4,4,84,15.417090,9.850693,-5.566397,-3.058587,case1_shapelet_static
